# NEUROSPORE — BCRA v1.0.0 Freeze & Release Notebook

This notebook does **not** tune or benchmark BCRA. It freezes the public definition of **BCRA v1.0.0**, verifies the full-connectome invariants, constructs the two validated operating regimes, and exports reproducibility artifacts for GitHub.

## Frozen family

- **BCRA-Native** — untouched retained FlyWire topology.
- **BCRA-Relaxed10** — 10% global destination-permutation relaxation.

## Frozen dynamics

`x[t+1] = 0.75*x[t] + 0.25*tanh(0.01*W@x[t] + input[t])`

The recurrent core is frozen during task-specific training. Task encoders/readouts may be trained.

The public Relaxed10 reference instance uses a fixed release seed only for reproducibility. It was **not selected from benchmark performance**.

Any architectural change after this notebook belongs to **BCRA v2** or a separately named experimental branch.


## 0. Runtime setup
For Colab, choose **Runtime → Change runtime type → GPU**. For Kaggle, enable a **GPU accelerator** in Notebook settings.

In [ ]:
import sys, subprocess, importlib.util
packages = {"pyarrow":"pyarrow", "scikit-learn":"sklearn", "pandas":"pandas", "numpy":"numpy", "matplotlib":"matplotlib"}
missing = [pip_name for pip_name, module_name in packages.items() if importlib.util.find_spec(module_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependencies ready.")

In [ ]:
import os, gc, json, random, re, urllib.request
from pathlib import Path
from dataclasses import dataclass
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

SEED=1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

ON_KAGGLE=os.path.exists('/kaggle/working')
ON_COLAB=os.path.exists('/content') and not ON_KAGGLE
ROOT=Path('/kaggle/working/neurospore_v0') if ON_KAGGLE else Path('/content/neurospore_v0') if ON_COLAB else Path.cwd()/'neurospore_v0'
DATA_RAW=ROOT/'data'/'raw'; DATA_PROCESSED=ROOT/'data'/'processed'; RESULTS=ROOT/'results'
for p in [DATA_RAW,DATA_PROCESSED,RESULTS]: p.mkdir(parents=True, exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Environment:', 'Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'Local/Jupyter')
print('Root:', ROOT); print('Torch:', torch.__version__); print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory/1e9,2))

## 1. Configuration
Set `max_neurons=20000` while debugging. Change to `None` for the full graph once the pipeline works.

In [ ]:
@dataclass
class Config:
    snapshot:int=783
    connectivity_file:str='connections_princeton.csv.gz'
    neuron_file:str='neurons.csv.gz'
    classification_file:str='classification.csv.gz'
    cell_types_file:str='consolidated_cell_types.csv.gz'
    max_neurons:int|None=None  # full retained FlyWire graph
    beta:float=0.95
    threshold:float=1.0
    reset_value:float=0.0
    refractory_steps:int=2
    weight_scale:float=0.030
    max_abs_edge_weight:float=1.0
    sim_steps:int=120
    pulse_start:int=10
    pulse_duration:int=5
    pulse_amplitude:float=1.5
    input_population_size:int=50
    sample_state_neurons:int=2048
CFG=Config(); CFG

## 2. Download / locate FAFB v783 data
This notebook uses the public FlyWire/Codex static release files. If automatic download fails, download the same files from the Codex FAFB Download Data page and upload them into `DATA_RAW`.

Required here:
- `connections_princeton.csv.gz`
- `neurons.csv.gz`
- `classification.csv.gz`
- `consolidated_cell_types.csv.gz`

In [ ]:
BASE=f'https://storage.googleapis.com/flywire-data/codex/data/fafb/{CFG.snapshot}'
FILES=[CFG.connectivity_file,CFG.neuron_file,CFG.classification_file,CFG.cell_types_file]

def download_file(url,dest,chunk_mb=8):
    if dest.exists() and dest.stat().st_size>0:
        print(f'✓ exists: {dest.name} ({dest.stat().st_size/1e6:.1f} MB)'); return True
    print('↓',dest.name)
    try:
        req=urllib.request.Request(url,headers={'User-Agent':'NEUROSPORE-v0 research notebook'})
        with urllib.request.urlopen(req,timeout=60) as r, open(dest,'wb') as f:
            while True:
                chunk=r.read(chunk_mb*1024*1024)
                if not chunk: break
                f.write(chunk)
        print(f'✓ downloaded: {dest.name} ({dest.stat().st_size/1e6:.1f} MB)'); return True
    except Exception as e:
        print('✗ failed:',e)
        if dest.exists(): dest.unlink(missing_ok=True)
        return False

status={name:download_file(f'{BASE}/{name}',DATA_RAW/name) for name in FILES}
missing=[k for k,v in status.items() if not v]
if missing:
    print('Manual fallback required for:',missing)
    print('Put the files in:',DATA_RAW)

In [ ]:
# Optional Colab manual upload helper. Change False -> True only if needed.
if False and ON_COLAB:
    from google.colab import files
    uploaded=files.upload()
    for name,blob in uploaded.items():
        (DATA_RAW/name).write_bytes(blob)
        print('saved',DATA_RAW/name)

## 3. Load and inspect schemas

In [ ]:
def read_csv_gz(path,**kwargs):
    if not path.exists(): raise FileNotFoundError(f'{path.name} is missing')
    return pd.read_csv(path,compression='gzip',low_memory=False,**kwargs)

neurons=read_csv_gz(DATA_RAW/CFG.neuron_file)
classification=read_csv_gz(DATA_RAW/CFG.classification_file)
cell_types=read_csv_gz(DATA_RAW/CFG.cell_types_file)
print('neurons',neurons.shape,list(neurons.columns)); display(neurons.head(3))
print('classification',classification.shape,list(classification.columns)); display(classification.head(3))
print('cell_types',cell_types.shape,list(cell_types.columns)); display(cell_types.head(3))

## 4. Load directed connectivity and normalize columns

In [ ]:
connections=read_csv_gz(DATA_RAW/CFG.connectivity_file)
print('raw connections',connections.shape,list(connections.columns)); display(connections.head(3))

def first_existing(df,candidates):
    for c in candidates:
        if c in df.columns: return c
    raise KeyError(f'None found: {candidates}')
PRE=first_existing(connections,['pre_root_id','pre_pt_root_id','source','pre'])
POST=first_existing(connections,['post_root_id','post_pt_root_id','target','post'])
SYN=first_existing(connections,['syn_count','weight','n_synapses','count'])
keep=[PRE,POST,SYN]
edges=connections[keep].copy()
edges[PRE]=edges[PRE].astype('int64'); edges[POST]=edges[POST].astype('int64')
edges[SYN]=pd.to_numeric(edges[SYN],errors='coerce').fillna(0).astype('float32')
edges=edges[edges[SYN]>0]
print('using',PRE,POST,SYN,'rows',len(edges))

## 5. Merge neuron metadata

In [ ]:
def root_col(df): return first_existing(df,['root_id','pt_root_id','id','root'])
nr=root_col(neurons); meta=neurons.copy(); meta[nr]=meta[nr].astype('int64')
cr=root_col(classification); cls=classification.copy(); cls[cr]=cls[cr].astype('int64'); cls=cls.rename(columns={cr:nr})
meta=meta.merge(cls,on=nr,how='left',suffixes=('','_class'))
tr=root_col(cell_types); ct=cell_types.copy(); ct[tr]=ct[tr].astype('int64'); ct=ct.rename(columns={tr:nr})
meta=meta.merge(ct,on=nr,how='left',suffixes=('','_ctype'))
print('merged metadata',meta.shape); display(meta.head(3))

## 6. Build simulation neuron set and contiguous IDs

In [ ]:
pair_edges=(edges.groupby([PRE,POST],as_index=False)[SYN].sum().rename(columns={SYN:'syn_count'}))
all_ids=np.union1d(pair_edges[PRE].unique(),pair_edges[POST].unique()).astype(np.int64)
if CFG.max_neurons is not None and len(all_ids)>CFG.max_neurons:
    pre_strength=pair_edges.groupby(PRE)['syn_count'].sum(); post_strength=pair_edges.groupby(POST)['syn_count'].sum()
    strength=pre_strength.add(post_strength,fill_value=0).sort_values(ascending=False)
    selected_ids=strength.head(CFG.max_neurons).index.to_numpy(dtype=np.int64)
else:
    selected_ids=all_ids
selected_set=set(selected_ids.tolist())
pair_edges=pair_edges[pair_edges[PRE].isin(selected_set)&pair_edges[POST].isin(selected_set)].copy()
selected_ids=np.sort(selected_ids)
root_to_idx={int(root):i for i,root in enumerate(selected_ids)}
pair_edges['src']=pair_edges[PRE].map(root_to_idx).astype(np.int64)
pair_edges['dst']=pair_edges[POST].map(root_to_idx).astype(np.int64)
N=len(selected_ids); E=len(pair_edges)
print(f'neurons={N:,} pair_edges={E:,} mean_edges/neuron={E/max(N,1):.2f}')

## 7. Provisional neurotransmitter sign policy
This is an engineering placeholder, not biological ground truth. ACh is positive, GABA negative, glutamate negative as a provisional fly-CNS simplification, and modulatory/unknown classes get a weak positive proxy. We will revise this after the first stable run.

In [ ]:
candidate_nt=['nt_type','predicted_nt','neurotransmitter','top_nt','nt_type_class']
NT_META=next((c for c in candidate_nt if c in meta.columns),None)
meta_sim=pd.DataFrame({nr:selected_ids})
if NT_META:
    lookup=meta[[nr,NT_META]].drop_duplicates(subset=[nr]); meta_sim=meta_sim.merge(lookup,on=nr,how='left')
else:
    NT_META='nt_type'; meta_sim[NT_META]='UNKNOWN'

def nt_sign(x):
    s=str(x).strip().upper()
    if 'ACH' in s or 'ACETYL' in s: return 1.0
    if 'GABA' in s: return -1.0
    if 'GLUT' in s or s=='GLU': return -1.0
    if any(k in s for k in ['DOP','SER','OCT']): return 0.25
    return 0.25
sign=np.array([nt_sign(x) for x in meta_sim[NT_META]],dtype=np.float32)
src_np=pair_edges['src'].to_numpy(np.int64); dst_np=pair_edges['dst'].to_numpy(np.int64)
syn_np=pair_edges['syn_count'].to_numpy(np.float32)
mag=np.log1p(syn_np).astype(np.float32)
weight_np=mag*sign[src_np]*CFG.weight_scale
weight_np=np.clip(weight_np,-CFG.max_abs_edge_weight,CFG.max_abs_edge_weight)
print('NT column:',NT_META,'weight range',weight_np.min(),weight_np.max(),'mean |w|',np.abs(weight_np).mean())

## 8. Build sparse recurrent matrix `W[post, pre]`

In [ ]:
src=torch.from_numpy(src_np).long(); dst=torch.from_numpy(dst_np).long(); weights=torch.from_numpy(weight_np).float()
indices=torch.stack([dst,src],dim=0)
W=torch.sparse_coo_tensor(indices,weights,size=(N,N),dtype=torch.float32).coalesce().to(DEVICE)
print(W); print('nnz',W._nnz())
del connections,edges; gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

## 9. Find candidate biological input/output populations
The search is deliberately transparent. Inspect results before treating them as canonical projection-neuron / MBON / descending-neuron populations.

In [ ]:
meta_indexed=meta[meta[nr].isin(selected_set)].copy(); meta_indexed['_sim_idx']=meta_indexed[nr].map(root_to_idx)
text_cols=[c for c in meta_indexed.columns if meta_indexed[c].dtype==object or pd.api.types.is_string_dtype(meta_indexed[c])]

def search_metadata(pattern,max_rows=25):
    mask=np.zeros(len(meta_indexed),dtype=bool)
    for c in text_cols:
        vals=meta_indexed[c].fillna('').astype(str)
        mask |= vals.str.contains(pattern,case=False,regex=True,na=False).to_numpy()
    show=[nr,'_sim_idx']+text_cols[:12]
    result=meta_indexed.loc[mask,show].drop_duplicates(subset=[nr])
    print(f'matches /{pattern}/:',len(result)); display(result.head(max_rows)); return result

pn_candidates=search_metadata(r'\bPN\b|projection neuron|olfactory|antennal')
mbon_candidates=search_metadata(r'\bMBON\b|mushroom body output')
dn_candidates=search_metadata(r'\bDN\b|descending neuron')

In [ ]:
def choose_indices(candidate_df,n,fallback):
    if candidate_df is not None and len(candidate_df)>=min(5,n):
        arr=candidate_df['_sim_idx'].dropna().astype(int).unique(); return np.array(arr[:n],dtype=np.int64),True
    return np.array(fallback[:min(n,len(fallback))],dtype=np.int64),False
strength_idx=(pair_edges.groupby('src')['syn_count'].sum().sort_values(ascending=False).index.to_numpy(np.int64))
INPUT_IDX,input_is_bio=choose_indices(pn_candidates,CFG.input_population_size,strength_idx)
out_candidates=pd.concat([mbon_candidates,dn_candidates],ignore_index=True)
OUTPUT_IDX,output_is_bio=choose_indices(out_candidates,100,strength_idx)
print('input',len(INPUT_IDX),'biological annotations',input_is_bio)
print('output',len(OUTPUT_IDX),'biological annotations',output_is_bio)

## Frozen BCRA v1 specification


In [ ]:
from dataclasses import dataclass, asdict
import hashlib, datetime

@dataclass(frozen=True)
class BCRAv1Spec:
    name: str = "BCRA"
    long_name: str = "Bio-Connectome Recurrent Architecture"
    version: str = "1.0.0"
    status: str = "FROZEN"
    flywire_snapshot: str = "783"
    expected_neurons: int = 138584
    expected_directed_edges: int = 3732460
    matrix_orientation: str = "W[post, pre]"
    edge_magnitude: str = "log1p(synapse_count)"
    neurotransmitter_sign_policy: str = "provisional BCRA v1 sign policy retained from validated experiments"
    biological_weight_scale: float = 0.5
    max_abs_edge_weight: float = 8.0
    dynamics: str = "globally_uniform_leaky_tanh"
    leak: float = 0.25
    recurrent_gain: float = 0.01
    self_memory: float = 0.75
    recurrent_core_trainable: bool = False
    task_encoder_trainable: bool = True
    task_readout_trainable: bool = True
    native_regime: str = "BCRA-Native"
    relaxed_regime: str = "BCRA-Relaxed10"
    relaxed_fraction: float = 0.10
    relaxed_transform: str = "select 10% of directed edge records and permute destinations; preserve selected destination multiset pre-coalesce"
    release_seed: int = 20260919

SPEC = BCRAv1Spec()

assert str(CFG.snapshot) == SPEC.flywire_snapshot
assert N == SPEC.expected_neurons
assert len(src_np) == SPEC.expected_directed_edges

print(json.dumps(asdict(SPEC), indent=2))
print("\nFull-graph invariants verified.")


## Canonical recurrent matrices


In [ ]:
def bcra_v1_biological_values():
    w = mag * sign[src_np] * SPEC.biological_weight_scale
    return np.clip(w, -SPEC.max_abs_edge_weight, SPEC.max_abs_edge_weight).astype(np.float32)

def bcra_sparse(use_src, use_dst, values):
    idx = torch.stack([
        torch.from_numpy(np.asarray(use_dst, dtype=np.int64)).long(),
        torch.from_numpy(np.asarray(use_src, dtype=np.int64)).long(),
    ], dim=0)
    return torch.sparse_coo_tensor(
        idx,
        torch.from_numpy(np.asarray(values, dtype=np.float32)),
        size=(N, N),
        dtype=torch.float32,
    ).coalesce().to(DEVICE)

BCRA_V1_VALUES = bcra_v1_biological_values()
W_BCRA_NATIVE = bcra_sparse(src_np, dst_np, BCRA_V1_VALUES)

assert W_BCRA_NATIVE.shape == (138584, 138584)
assert W_BCRA_NATIVE._nnz() == 3732460
assert np.isfinite(BCRA_V1_VALUES).all()

print("BCRA-Native ready:", W_BCRA_NATIVE.shape, "nnz=", W_BCRA_NATIVE._nnz())


## Canonical BCRA-Relaxed10 reference transformation


In [ ]:
def make_relaxed10_destination_map(seed=SPEC.release_seed):
    rng = np.random.default_rng(int(seed))
    n_edges = len(dst_np)
    m = int(round(SPEC.relaxed_fraction * n_edges))
    chosen = rng.choice(n_edges, size=m, replace=False)

    relaxed_dst = np.asarray(dst_np, dtype=np.int64).copy()
    original_selected = relaxed_dst[chosen].copy()
    relaxed_dst[chosen] = rng.permutation(relaxed_dst[chosen])

    meta = {
        "seed": int(seed),
        "requested_fraction": float(SPEC.relaxed_fraction),
        "selected_edge_records": int(m),
        "selected_fraction": float(m / n_edges),
        "actual_changed_endpoint_fraction": float(np.mean(relaxed_dst != dst_np)),
        "selected_destination_multiset_preserved_pre_coalesce": bool(
            np.array_equal(np.sort(original_selected), np.sort(relaxed_dst[chosen]))
        ),
    }
    return relaxed_dst, chosen.astype(np.int64), meta

RELAXED10_DST, RELAXED10_SELECTED, RELAXED10_META = make_relaxed10_destination_map()

assert RELAXED10_META["selected_destination_multiset_preserved_pre_coalesce"]
assert abs(RELAXED10_META["selected_fraction"] - 0.10) < 1e-6
assert 0.095 <= RELAXED10_META["actual_changed_endpoint_fraction"] <= 0.105

W_BCRA_RELAXED10 = bcra_sparse(src_np, RELAXED10_DST, BCRA_V1_VALUES)
RELAXED10_META["nnz_after_coalesce"] = int(W_BCRA_RELAXED10._nnz())

print(json.dumps(RELAXED10_META, indent=2))


## Frozen recurrent dynamics


In [ ]:
class BCRAv1LeakyTanh:
    def __init__(self, W):
        self.W = W
        self.leak = float(SPEC.leak)
        self.gain = float(SPEC.recurrent_gain)

    @torch.no_grad()
    def step(self, x, input_t):
        candidate = torch.tanh(self.gain * torch.sparse.mm(self.W, x) + input_t)
        return (1.0 - self.leak) * x + self.leak * candidate

    @torch.no_grad()
    def run(self, current_TNB, record_indices=None):
        T, _, B = current_TNB.shape
        x = torch.zeros(N, B, device=DEVICE)

        if record_indices is None:
            for t in range(T):
                x = self.step(x, current_TNB[t])
            return x

        rec = torch.as_tensor(record_indices, device=DEVICE, dtype=torch.long)
        history = []
        for t in range(T):
            x = self.step(x, current_TNB[t])
            history.append(x[rec, :].T.cpu())
        return torch.stack(history, dim=1)

BCRA_NATIVE = BCRAv1LeakyTanh(W_BCRA_NATIVE)
BCRA_RELAXED10 = BCRAv1LeakyTanh(W_BCRA_RELAXED10)

assert BCRA_NATIVE.leak == 0.25
assert BCRA_NATIVE.gain == 0.01
assert abs((1.0 - BCRA_NATIVE.leak) - 0.75) < 1e-12

print("Frozen BCRA v1 recurrent update verified.")


## Deterministic smoke test


In [ ]:
@torch.no_grad()
def freeze_smoke_test(sim, seed):
    rng = np.random.default_rng(int(seed))
    chosen = rng.choice(
        np.asarray(INPUT_IDX, dtype=np.int64),
        size=min(8, len(INPUT_IDX)),
        replace=False
    )

    T, B = 12, 2
    cur = torch.zeros(T, N, B, device=DEVICE)
    idx = torch.as_tensor(chosen, device=DEVICE, dtype=torch.long)
    cur[2:5, idx, :] = 1.0

    record = np.asarray(OUTPUT_IDX[:min(16, len(OUTPUT_IDX))], dtype=np.int64)
    hist = sim.run(cur, record_indices=record)

    assert hist.shape == (B, T, len(record))
    assert torch.isfinite(hist).all()

    return {
        "shape": list(hist.shape),
        "finite": True,
        "absolute_sum": float(hist.abs().sum().item()),
    }

SMOKE_NATIVE = freeze_smoke_test(BCRA_NATIVE, 111)
SMOKE_RELAXED10 = freeze_smoke_test(BCRA_RELAXED10, 222)

print("Native:", SMOKE_NATIVE)
print("Relaxed10:", SMOKE_RELAXED10)


## Freeze invariants


In [ ]:
FREEZE_CHECKS = {
    "version_is_1_0_0": SPEC.version == "1.0.0",
    "status_frozen": SPEC.status == "FROZEN",
    "snapshot_v783": str(CFG.snapshot) == "783",
    "neuron_count_exact": N == 138584,
    "edge_count_exact": len(src_np) == 3732460,
    "matrix_orientation_post_pre": SPEC.matrix_orientation == "W[post, pre]",
    "biological_scale_0_5": SPEC.biological_weight_scale == 0.5,
    "leak_0_25": SPEC.leak == 0.25,
    "self_memory_0_75": SPEC.self_memory == 0.75,
    "gain_0_01": SPEC.recurrent_gain == 0.01,
    "core_frozen": SPEC.recurrent_core_trainable is False,
    "relaxed_fraction_0_10": SPEC.relaxed_fraction == 0.10,
    "relaxed_multiset_preserved": RELAXED10_META["selected_destination_multiset_preserved_pre_coalesce"],
    "native_smoke_finite": SMOKE_NATIVE["finite"],
    "relaxed10_smoke_finite": SMOKE_RELAXED10["finite"],
}

failed = [k for k, v in FREEZE_CHECKS.items() if not bool(v)]
print(json.dumps(FREEZE_CHECKS, indent=2))

if failed:
    raise AssertionError("BCRA v1 freeze invariant failure: " + ", ".join(failed))

print("\n" + "=" * 72)
print("BCRA v1.0.0 ARCHITECTURE FREEZE VERIFIED")
print("=" * 72)


## Export GitHub release artifacts


In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)

config_path = RESULTS / "bcra_v1_reference_config.json"
config_path.write_text(json.dumps(asdict(SPEC), indent=2))

patch_path = RESULTS / "bcra_v1_relaxed10_reference_patch.npz"
np.savez_compressed(
    patch_path,
    selected_edge_indices=RELAXED10_SELECTED.astype(np.int64),
    replacement_destinations=RELAXED10_DST[RELAXED10_SELECTED].astype(np.int64),
    release_seed=np.asarray([SPEC.release_seed], dtype=np.int64),
)

arch_lines = [
    "# BCRA v1.0.0 — Frozen Architecture Specification",
    "",
    "**BCRA**: Bio-Connectome Recurrent Architecture",
    "",
    "Status: **FROZEN**",
    "",
    "## Connectome",
    "",
    f"- FlyWire FAFB snapshot: v{SPEC.flywire_snapshot}",
    f"- Retained neurons: {SPEC.expected_neurons:,}",
    f"- Directed edges: {SPEC.expected_directed_edges:,}",
    f"- Matrix orientation: `{SPEC.matrix_orientation}`",
    f"- Edge magnitude: `{SPEC.edge_magnitude}`",
    f"- Biological weight scale: `{SPEC.biological_weight_scale}`",
    "",
    "## Recurrent dynamics",
    "",
    "```text",
    "x[t+1] = 0.75*x[t] + 0.25*tanh(0.01*W@x[t] + input[t])",
    "```",
    "",
    "The recurrent core is frozen during task-specific training.",
    "",
    "## Operating regimes",
    "",
    "### BCRA-Native",
    "Untouched retained FlyWire topology.",
    "",
    "### BCRA-Relaxed10",
    "Select 10% of directed edge records and permute destinations among those selected records.",
    "The selected destination multiset is preserved pre-coalesce.",
    "",
    f"Public reference release seed: `{SPEC.release_seed}`.",
    "",
    "The seed exists only for exact reproducibility and was not selected from benchmark performance.",
    "",
    "## Versioning",
    "",
    "Any architectural change belongs to BCRA v2 or a separately named experimental branch.",
]
arch_path = RESULTS / "BCRA_v1_ARCHITECTURE.md"
arch_path.write_text("\n".join(arch_lines))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = {
    "name": SPEC.name,
    "long_name": SPEC.long_name,
    "version": SPEC.version,
    "status": SPEC.status,
    "frozen_at_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "spec": asdict(SPEC),
    "graph": {
        "neurons": int(N),
        "directed_edge_records": int(len(src_np)),
        "native_sparse_nnz": int(W_BCRA_NATIVE._nnz()),
        "relaxed10_sparse_nnz": int(W_BCRA_RELAXED10._nnz()),
    },
    "relaxed10_reference": RELAXED10_META,
    "smoke_tests": {
        "native": SMOKE_NATIVE,
        "relaxed10": SMOKE_RELAXED10,
    },
    "freeze_checks": FREEZE_CHECKS,
    "artifacts": {
        config_path.name: {"sha256": sha256_file(config_path)},
        patch_path.name: {"sha256": sha256_file(patch_path)},
        arch_path.name: {"sha256": sha256_file(arch_path)},
    },
    "versioning_rule": "BCRA v1.0.0 is immutable. Architectural changes belong to BCRA v2 or a separately named experimental branch.",
}

manifest_path = RESULTS / "BCRA_v1_FREEZE_MANIFEST.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

print("Exported:")
for p in [config_path, patch_path, arch_path, manifest_path]:
    print(" -", p)

print("\nBCRA v1.0.0 freeze complete.")


# Release checklist

After this notebook completes without assertion failures:

1. Keep this notebook in the GitHub repository as the canonical freeze notebook.
2. Commit the generated `bcra_v1_reference_config.json`, `bcra_v1_relaxed10_reference_patch.npz`, `BCRA_v1_ARCHITECTURE.md`, and `BCRA_v1_FREEZE_MANIFEST.json`.
3. Tag the release as `bcra-v1.0.0`.
4. Do not modify the v1 architecture definition afterward.
5. Continue architecture development under **BCRA v2**.
